In [3]:
from pathlib import Path
import numpy as np
import tensorflow as tf

PROCESSED_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")

print("TensorFlow version:", tf.__version__)
print("Processed directory:", PROCESSED_DIR)
print("Model directory:", MODEL_DIR)

TensorFlow version: 2.21.0
Processed directory: ..\data\processed
Model directory: ..\models


In [4]:
# Load baseline LSTM

model = tf.keras.models.load_model(
    MODEL_DIR / "baseline_lstm.keras"
)

# Load clean test data
X_test_lstm = np.load(
    PROCESSED_DIR / "X_test_lstm.npy"
)

y_test_lstm = np.load(
    PROCESSED_DIR / "y_test_lstm.npy"
)

print("Model loaded successfully")
print("Test data shape :", X_test_lstm.shape)
print("Test labels shape:", y_test_lstm.shape)

Model loaded successfully
Test data shape : (45000, 78, 1)
Test labels shape: (45000,)


In [5]:
# Load adversarial test data

X_test_fgsm = np.load(
    PROCESSED_DIR / "X_test_fgsm_eps001.npy"
)

X_test_pgd = np.load(
    PROCESSED_DIR / "X_test_pgd_eps001.npy"
)

X_test_deepfool = np.load(
    PROCESSED_DIR / "X_test_deepfool.npy"
)

print("FGSM      :", X_test_fgsm.shape)
print("PGD       :", X_test_pgd.shape)
print("DeepFool  :", X_test_deepfool.shape)

FGSM      : (45000, 78, 1)
PGD       : (45000, 78, 1)
DeepFool  : (45000, 78, 1)


In [6]:
%pip install shap

Note: you may need to restart the kernel to use updated packages.


In [7]:
import shap
print("SHAP version:", shap.__version__)

SHAP version: 0.52.0


In [10]:
# Create SHAP GradientExplainer

explainer = shap.GradientExplainer(
    model,
    background
)

print("SHAP GradientExplainer created successfully.")

SHAP GradientExplainer created successfully.


In [11]:
# Generate SHAP values for 10 samples

shap_values = explainer.shap_values(
    X_shap_test
)

print("SHAP calculation completed.")
print("Type:", type(shap_values))

c:\SentinelXAI\.venv\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_2
Received: inputs=['Tensor(shape=(10, 78, 1))']
  warnings.warn(msg)
c:\SentinelXAI\.venv\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_2
Received: inputs=['Tensor(shape=(50, 78, 1))']
  warnings.warn(msg)


SHAP calculation completed.
Type: <class 'numpy.ndarray'>


In [12]:
print("SHAP values shape:", shap_values.shape)
print("SHAP values dtype:", shap_values.dtype)
print("Number of dimensions:", shap_values.ndim)
# Check the magnitude of the SHAP values

print("Minimum SHAP value :", np.min(shap_values))
print("Maximum SHAP value :", np.max(shap_values))
print("Mean absolute SHAP :", np.mean(np.abs(shap_values)))

SHAP values shape: (10, 78, 1, 1)
SHAP values dtype: float64
Number of dimensions: 4
Minimum SHAP value : -0.5425400733947754
Maximum SHAP value : 0.42090052366256714
Mean absolute SHAP : 0.02313934048301137


In [13]:
# Convert SHAP values to a feature-level matrix

shap_values_features = np.squeeze(
    shap_values,
    axis=(2, 3)
)

print("Original SHAP shape :", shap_values.shape)
print("Feature SHAP shape  :", shap_values_features.shape)

Original SHAP shape : (10, 78, 1, 1)
Feature SHAP shape  : (10, 78)


In [14]:
# Verify feature-level SHAP values

print("Minimum SHAP value :", np.min(shap_values_features))
print("Maximum SHAP value :", np.max(shap_values_features))
print("Mean absolute SHAP :", np.mean(np.abs(shap_values_features)))

print()
print("First sample SHAP values:")
print(shap_values_features[0])

Minimum SHAP value : -0.5425400733947754
Maximum SHAP value : 0.42090052366256714
Mean absolute SHAP : 0.02313934048301137

First sample SHAP values:
[ 5.71562946e-02  1.24387071e-01  2.96591874e-03 -2.14732205e-03
  2.57948879e-03 -4.03919304e-03  2.58184057e-02  2.47040372e-02
  8.32552323e-04 -9.68960114e-03  1.31515600e-02 -3.29804495e-02
  2.04931110e-01  1.58963934e-01  8.34416947e-04 -1.42780854e-03
 -1.89553872e-02 -1.74591485e-02 -1.10679055e-02 -8.85175541e-05
 -2.13440377e-02 -1.06321583e-02 -1.06763057e-02 -8.56259558e-03
 -3.48416012e-04 -1.09670814e-02 -2.90112384e-03 -2.28729192e-03
 -2.17100326e-03  7.01454293e-04  1.86740775e-02  0.00000000e+00
  0.00000000e+00  0.00000000e+00 -5.94793903e-08 -5.22647525e-08
 -4.53738612e-04 -4.03909275e-04 -1.58841535e-02  5.19299507e-02
  9.20245275e-02  7.31847212e-02  5.97786605e-02  0.00000000e+00
  8.55067652e-03  0.00000000e+00  3.53939869e-02 -1.57132745e-02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  1.14124678e-02
 -4.7

In [15]:
# Select the same 10 samples for each attack

X_shap_fgsm = X_test_fgsm[:10]
X_shap_pgd = X_test_pgd[:10]
X_shap_deepfool = X_test_deepfool[:10]

print("Clean      :", X_shap_test.shape)
print("FGSM       :", X_shap_fgsm.shape)
print("PGD        :", X_shap_pgd.shape)
print("DeepFool   :", X_shap_deepfool.shape)

Clean      : (10, 78, 1)
FGSM       : (10, 78, 1)
PGD        : (10, 78, 1)
DeepFool   : (10, 78, 1)


In [16]:
# Generate SHAP explanations for FGSM samples

shap_fgsm = explainer.shap_values(
    X_shap_fgsm
)

shap_fgsm_features = np.squeeze(
    shap_fgsm,
    axis=(2, 3)
)

print("FGSM SHAP shape:", shap_fgsm_features.shape)

FGSM SHAP shape: (10, 78)


In [17]:
# Generate SHAP explanations for PGD samples

shap_pgd = explainer.shap_values(
    X_shap_pgd
)

shap_pgd_features = np.squeeze(
    shap_pgd,
    axis=(2, 3)
)

print("PGD SHAP shape:", shap_pgd_features.shape)

PGD SHAP shape: (10, 78)


In [18]:
# Generate SHAP explanations for DeepFool samples

shap_deepfool = explainer.shap_values(
    X_shap_deepfool
)

shap_deepfool_features = np.squeeze(
    shap_deepfool,
    axis=(2, 3)
)

print("DeepFool SHAP shape:", shap_deepfool_features.shape)

DeepFool SHAP shape: (10, 78)


In [19]:
# Create the matched SHAP analysis subset

SHAP_SAMPLES = 100

X_shap_clean = X_test_lstm[:SHAP_SAMPLES]
X_shap_fgsm = X_test_fgsm[:SHAP_SAMPLES]
X_shap_pgd = X_test_pgd[:SHAP_SAMPLES]
X_shap_deepfool = X_test_deepfool[:SHAP_SAMPLES]

y_shap = y_test_lstm[:SHAP_SAMPLES]

print("SHAP analysis subset")
print("=" * 50)
print("Number of samples:", SHAP_SAMPLES)
print("Clean      :", X_shap_clean.shape)
print("FGSM       :", X_shap_fgsm.shape)
print("PGD        :", X_shap_pgd.shape)
print("DeepFool   :", X_shap_deepfool.shape)
print("Labels     :", y_shap.shape)

SHAP analysis subset
Number of samples: 100
Clean      : (100, 78, 1)
FGSM       : (100, 78, 1)
PGD        : (100, 78, 1)
DeepFool   : (100, 78, 1)
Labels     : (100,)


In [20]:
import time

# Generate SHAP explanations for the 100 clean samples

start_time = time.time()

shap_clean_100 = explainer.shap_values(
    X_shap_clean
)

elapsed_time = time.time() - start_time

shap_clean_100_features = np.squeeze(
    shap_clean_100,
    axis=(2, 3)
)

print("Clean SHAP calculation completed")
print("=" * 50)
print("SHAP shape       :", shap_clean_100_features.shape)
print(f"Time taken       : {elapsed_time:.2f} seconds")
print(f"Time per sample  : {elapsed_time / SHAP_SAMPLES:.2f} seconds")

c:\SentinelXAI\.venv\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_2
Received: inputs=['Tensor(shape=(100, 78, 1))']
  warnings.warn(msg)


Clean SHAP calculation completed
SHAP shape       : (100, 78)
Time taken       : 6.27 seconds
Time per sample  : 0.06 seconds


actual implementation of shap

In [21]:
# Create a reproducible 500-sample SHAP analysis subset

SHAP_SAMPLES = 500
SHAP_RANDOM_STATE = 42

rng = np.random.default_rng(SHAP_RANDOM_STATE)

shap_indices = rng.choice(
    len(X_test_lstm),
    size=SHAP_SAMPLES,
    replace=False
)

shap_indices = np.sort(shap_indices)

X_shap_clean = X_test_lstm[shap_indices]
X_shap_fgsm = X_test_fgsm[shap_indices]
X_shap_pgd = X_test_pgd[shap_indices]
X_shap_deepfool = X_test_deepfool[shap_indices]

y_shap = y_test_lstm[shap_indices]

print("SHAP analysis subset")
print("=" * 50)
print("Samples:", SHAP_SAMPLES)
print("Random state:", SHAP_RANDOM_STATE)

print()
print("Clean      :", X_shap_clean.shape)
print("FGSM       :", X_shap_fgsm.shape)
print("PGD        :", X_shap_pgd.shape)
print("DeepFool   :", X_shap_deepfool.shape)
print("Labels     :", y_shap.shape)

SHAP analysis subset
Samples: 500
Random state: 42

Clean      : (500, 78, 1)
FGSM       : (500, 78, 1)
PGD        : (500, 78, 1)
DeepFool   : (500, 78, 1)
Labels     : (500,)


In [22]:
# Generate SHAP explanations for the 500 matched samples

print("Generating SHAP explanations...")

# Clean
shap_clean_raw = explainer.shap_values(X_shap_clean)
print("Clean SHAP generated.")

# FGSM
shap_fgsm_raw = explainer.shap_values(X_shap_fgsm)
print("FGSM SHAP generated.")

# PGD
shap_pgd_raw = explainer.shap_values(X_shap_pgd)
print("PGD SHAP generated.")

# DeepFool
shap_deepfool_raw = explainer.shap_values(X_shap_deepfool)
print("DeepFool SHAP generated.")

Generating SHAP explanations...


c:\SentinelXAI\.venv\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer_2
Received: inputs=['Tensor(shape=(500, 78, 1))']
  warnings.warn(msg)


Clean SHAP generated.
FGSM SHAP generated.
PGD SHAP generated.
DeepFool SHAP generated.


In [24]:
# Convert SHAP outputs to (500, 78)

def process_shap_values(shap_values):
    shap_values = np.asarray(shap_values)
    return np.squeeze(shap_values)


shap_clean = process_shap_values(shap_clean_raw)
shap_fgsm = process_shap_values(shap_fgsm_raw)
shap_pgd = process_shap_values(shap_pgd_raw)
shap_deepfool = process_shap_values(shap_deepfool_raw)

print("SHAP shapes:")
print("Clean:   ", shap_clean.shape)
print("FGSM:    ", shap_fgsm.shape)
print("PGD:     ", shap_pgd.shape)
print("DeepFool:", shap_deepfool.shape)

SHAP shapes:
Clean:    (500, 78)
FGSM:     (500, 78)
PGD:      (500, 78)
DeepFool: (500, 78)


In [25]:
# Calculate Explanation Drift
# Explanation Drift = mean absolute change in SHAP values
# between clean and adversarial samples.

explanation_drift_fgsm = np.mean(
    np.abs(shap_clean - shap_fgsm),
    axis=1
)

explanation_drift_pgd = np.mean(
    np.abs(shap_clean - shap_pgd),
    axis=1
)

explanation_drift_deepfool = np.mean(
    np.abs(shap_clean - shap_deepfool),
    axis=1
)

print("Explanation Drift calculated.")
print()

print("FGSM:")
print("  Mean:", explanation_drift_fgsm.mean())
print("  Median:", np.median(explanation_drift_fgsm))
print("  Min:", explanation_drift_fgsm.min())
print("  Max:", explanation_drift_fgsm.max())

print("\nPGD:")
print("  Mean:", explanation_drift_pgd.mean())
print("  Median:", np.median(explanation_drift_pgd))
print("  Min:", explanation_drift_pgd.min())
print("  Max:", explanation_drift_pgd.max())

print("\nDeepFool:")
print("  Mean:", explanation_drift_deepfool.mean())
print("  Median:", np.median(explanation_drift_deepfool))
print("  Min:", explanation_drift_deepfool.min())
print("  Max:", explanation_drift_deepfool.max())

Explanation Drift calculated.

FGSM:
  Mean: 0.015861150068147745
  Median: 0.013424840744347667
  Min: 0.003704073321002914
  Max: 0.09457938177411056

PGD:
  Mean: 0.016221494912864298
  Median: 0.013490239018535343
  Min: 0.0023916816341616877
  Max: 0.11132507704123995

DeepFool:
  Mean: 0.02778378118666395
  Median: 0.025643384753412678
  Min: 0.007640777232618818
  Max: 0.09129721852959022


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between clean and adversarial SHAP explanations

shap_similarity_fgsm = np.diag(
    cosine_similarity(shap_clean, shap_fgsm)
)

shap_similarity_pgd = np.diag(
    cosine_similarity(shap_clean, shap_pgd)
)

shap_similarity_deepfool = np.diag(
    cosine_similarity(shap_clean, shap_deepfool)
)

print("SHAP Similarity calculated.")
print()

print("FGSM:")
print("  Mean:", shap_similarity_fgsm.mean())
print("  Median:", np.median(shap_similarity_fgsm))
print("  Min:", shap_similarity_fgsm.min())
print("  Max:", shap_similarity_fgsm.max())

print("\nPGD:")
print("  Mean:", shap_similarity_pgd.mean())
print("  Median:", np.median(shap_similarity_pgd))
print("  Min:", shap_similarity_pgd.min())
print("  Max:", shap_similarity_pgd.max())

print("\nDeepFool:")
print("  Mean:", shap_similarity_deepfool.mean())
print("  Median:", np.median(shap_similarity_deepfool))
print("  Min:", shap_similarity_deepfool.min())
print("  Max:", shap_similarity_deepfool.max())

SHAP Similarity calculated.

FGSM:
  Mean: 0.6796035700244435
  Median: 0.7881504167783286
  Min: -0.8762390675628748
  Max: 0.9962182167585026

PGD:
  Mean: 0.6645008066571667
  Median: 0.7663238854963137
  Min: -0.9225228895704279
  Max: 0.9940305424875444

DeepFool:
  Mean: 0.3602441103878922
  Median: 0.3821509180057888
  Min: -0.8499351770690192
  Max: 0.9850083623853404


In [27]:
TOP_K = 10

def top_k_features(shap_values, k=TOP_K):
    # Indices of features with largest absolute SHAP values
    return np.argsort(np.abs(shap_values), axis=1)[:, -k:]


# Get top-10 features for each condition
top_clean = top_k_features(shap_clean)
top_fgsm = top_k_features(shap_fgsm)
top_pgd = top_k_features(shap_pgd)
top_deepfool = top_k_features(shap_deepfool)


def calculate_top_k_overlap(clean_top, adv_top):
    overlap = []

    for clean_features, adv_features in zip(clean_top, adv_top):
        common = len(set(clean_features) & set(adv_features))
        overlap.append(common / TOP_K)

    return np.array(overlap)


overlap_fgsm = calculate_top_k_overlap(top_clean, top_fgsm)
overlap_pgd = calculate_top_k_overlap(top_clean, top_pgd)
overlap_deepfool = calculate_top_k_overlap(top_clean, top_deepfool)


print("Top-10 Feature Overlap")
print()

print("FGSM:")
print("  Mean:", overlap_fgsm.mean())
print("  Median:", np.median(overlap_fgsm))

print("\nPGD:")
print("  Mean:", overlap_pgd.mean())
print("  Median:", np.median(overlap_pgd))

print("\nDeepFool:")
print("  Mean:", overlap_deepfool.mean())
print("  Median:", np.median(overlap_deepfool))

Top-10 Feature Overlap

FGSM:
  Mean: 0.6774
  Median: 0.7

PGD:
  Mean: 0.6775999999999999
  Median: 0.7

DeepFool:
  Mean: 0.4494
  Median: 0.4


In [28]:
# Get prediction probabilities for the 500 SHAP samples

clean_probs = model.predict(X_shap_clean, verbose=0).reshape(-1)
fgsm_probs = model.predict(X_shap_fgsm, verbose=0).reshape(-1)
pgd_probs = model.predict(X_shap_pgd, verbose=0).reshape(-1)
deepfool_probs = model.predict(X_shap_deepfool, verbose=0).reshape(-1)


# Model confidence = distance from 0.5 decision boundary
clean_confidence = np.abs(clean_probs - 0.5) * 2
fgsm_confidence = np.abs(fgsm_probs - 0.5) * 2
pgd_confidence = np.abs(pgd_probs - 0.5) * 2
deepfool_confidence = np.abs(deepfool_probs - 0.5) * 2


# Confidence drop relative to clean input
confidence_drop_fgsm = clean_confidence - fgsm_confidence
confidence_drop_pgd = clean_confidence - pgd_confidence
confidence_drop_deepfool = clean_confidence - deepfool_confidence


print("Confidence Drop")
print()

print("FGSM:")
print("  Mean:", confidence_drop_fgsm.mean())
print("  Median:", np.median(confidence_drop_fgsm))

print("\nPGD:")
print("  Mean:", confidence_drop_pgd.mean())
print("  Median:", np.median(confidence_drop_pgd))

print("\nDeepFool:")
print("  Mean:", confidence_drop_deepfool.mean())
print("  Median:", np.median(confidence_drop_deepfool))

Confidence Drop

FGSM:
  Mean: 0.023536352
  Median: 0.0008684993

PGD:
  Mean: 0.026820794
  Median: 0.00089272857

DeepFool:
  Mean: 0.6168139
  Median: 0.7816797


In [29]:
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path(r"C:\SentinelXAI\results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save SHAP explanations
np.save(RESULTS_DIR / "shap_clean_500.npy", shap_clean)
np.save(RESULTS_DIR / "shap_fgsm_500.npy", shap_fgsm)
np.save(RESULTS_DIR / "shap_pgd_500.npy", shap_pgd)
np.save(RESULTS_DIR / "shap_deepfool_500.npy", shap_deepfool)

# Create behavior feature table
behavior_df = pd.DataFrame({
    "sample_index": shap_indices,

    "fgsm_explanation_drift": explanation_drift_fgsm,
    "fgsm_shap_similarity": shap_similarity_fgsm,
    "fgsm_top10_overlap": overlap_fgsm,
    "fgsm_confidence_drop": confidence_drop_fgsm,

    "pgd_explanation_drift": explanation_drift_pgd,
    "pgd_shap_similarity": shap_similarity_pgd,
    "pgd_top10_overlap": overlap_pgd,
    "pgd_confidence_drop": confidence_drop_pgd,

    "deepfool_explanation_drift": explanation_drift_deepfool,
    "deepfool_shap_similarity": shap_similarity_deepfool,
    "deepfool_top10_overlap": overlap_deepfool,
    "deepfool_confidence_drop": confidence_drop_deepfool
})

behavior_path = RESULTS_DIR / "xai_behavior_features_500.csv"
behavior_df.to_csv(behavior_path, index=False)

print("Saved successfully.")
print()
print("SHAP files:")
print(" - shap_clean_500.npy")
print(" - shap_fgsm_500.npy")
print(" - shap_pgd_500.npy")
print(" - shap_deepfool_500.npy")
print()
print("Behavior table:")
print(behavior_path)
print()
print("Shape:", behavior_df.shape)

Saved successfully.

SHAP files:
 - shap_clean_500.npy
 - shap_fgsm_500.npy
 - shap_pgd_500.npy
 - shap_deepfool_500.npy

Behavior table:
C:\SentinelXAI\results\xai_behavior_features_500.csv

Shape: (500, 13)
